# Gold Payment Fact

This notebook builds the `fact_payments` Gold model from the cleaned Silver order payments dataset.

**Grain:** One row per `order_id` and `payment_sequential`.

In [0]:
from pyspark.sql import functions as F

## 1. Define storage paths

In [0]:
SILVER_ORDER_PAYMENTS_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/order_payments"
)

GOLD_FACT_PAYMENTS_PATH = (
    "abfss://gold@stnovacartdev.dfs.core.windows.net/"
    "olist/fact_payments"
)

print(f"Silver source: {SILVER_ORDER_PAYMENTS_PATH}")
print(f"Gold target: {GOLD_FACT_PAYMENTS_PATH}")

## 2. Read Silver order payments

In [0]:
silver_order_payments_df = (
    spark.read
    .format("delta")
    .load(SILVER_ORDER_PAYMENTS_PATH)
)

silver_payment_count = silver_order_payments_df.count()

print(f"Silver payment rows: {silver_payment_count:,}")

display(silver_order_payments_df.limit(10))

## 3. Validate required columns

In [0]:
required_columns = {
    "order_id",
    "payment_sequential",
    "payment_type",
    "payment_installments",
    "payment_value",
    "_silver_processed_at",
}

missing_columns = required_columns - set(silver_order_payments_df.columns)

if missing_columns:
    raise ValueError(
        "Silver order payments is missing required columns: "
        f"{sorted(missing_columns)}"
    )

print("Required column validation passed.")

## 4. Build payment fact

In [0]:
fact_payments_df = (
    silver_order_payments_df
    .select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value",
        "_silver_processed_at",
    )
    .withColumn(
        "is_installment_payment",
        F.col("payment_installments") > 1,
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

display(fact_payments_df.limit(10))

## 5. Validate payment fact

In [0]:
fact_payment_count = fact_payments_df.count()

duplicate_payment_count = (
    fact_payments_df
    .groupBy("order_id", "payment_sequential")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_grain_count = (
    fact_payments_df
    .filter(
        F.col("order_id").isNull()
        | F.col("payment_sequential").isNull()
    )
    .count()
)

invalid_payment_value_count = (
    fact_payments_df
    .filter(
        F.col("payment_value").isNull()
        | (F.col("payment_value") < 0)
    )
    .count()
)

invalid_installment_count = (
    fact_payments_df
    .filter(
        F.col("payment_installments").isNull()
        | (F.col("payment_installments") < 0)
    )
    .count()
)

if fact_payment_count == 0:
    raise ValueError("Payment fact is empty.")

if fact_payment_count != silver_payment_count:
    raise ValueError(
        "Payment fact row count does not match Silver payments. "
        f"Silver: {silver_payment_count:,}, "
        f"Gold: {fact_payment_count:,}"
    )

if duplicate_payment_count > 0:
    raise ValueError(
        "Payment fact contains "
        f"{duplicate_payment_count:,} duplicate grain combinations."
    )

if null_grain_count > 0:
    raise ValueError(
        f"Payment fact contains {null_grain_count:,} rows with null grain keys."
    )

if invalid_payment_value_count > 0:
    raise ValueError(
        "Payment fact contains "
        f"{invalid_payment_value_count:,} invalid payment values."
    )

if invalid_installment_count > 0:
    raise ValueError(
        "Payment fact contains "
        f"{invalid_installment_count:,} invalid installment values."
    )

print(f"Payment fact rows: {fact_payment_count:,}")
print("Payment fact grain validation passed.")

## 6. Write payment fact to Gold

In [0]:
(
    fact_payments_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_FACT_PAYMENTS_PATH)
)

print(f"Payment fact written to: {GOLD_FACT_PAYMENTS_PATH}")

## 7. Validate Gold output

In [0]:
written_fact_payments_df = (
    spark.read
    .format("delta")
    .load(GOLD_FACT_PAYMENTS_PATH)
)

written_payment_count = written_fact_payments_df.count()

if written_payment_count != fact_payment_count:
    raise ValueError(
        "Gold payment fact write validation failed. "
        f"Expected: {fact_payment_count:,}, "
        f"Written: {written_payment_count:,}"
    )

print(f"Written payment fact rows: {written_payment_count:,}")
print("Gold payment fact write validation passed.")

## 8. Inspect Gold payment fact

In [0]:
written_fact_payments_df.printSchema()

display(
    written_fact_payments_df
    .orderBy("order_id", "payment_sequential")
    .limit(10)
)